# 5주차 윤리 감사 실습, 데이터 편향을 코드로 확인하기

**AI교육의 이해 / 언플러그드AI교육, Chapter 5 인공지능 윤리**
전인성, 광주교육대학교 컴퓨터교육과

<a href="https://colab.research.google.com/github/jinsung4069/jinsung4069.github.io/blob/main/data/lectures/ai-education-ch5-bias-lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 상황
학급 활동지에 적은 손글씨 숫자를 사진으로 찍어 자동으로 읽는 도구를 도입하려고 합니다. 판매 자료에는 **정확도 95%** 라고 적혀 있습니다. 그런데 우리 교실은 오후에 어둡고, 어두운 교실에서 찍은 사진은 흐리고 잡음이 많습니다.

## 연구 질문
어두운 사진이 거의 없는 학습 자료로 만든 모델은 어두운 사진에서 오류가 늘어날까요? 학습 자료에 어두운 사진을 충분히 포함하면 차이가 줄어들까요?

## 실습 방법
1. 위에서부터 코드 셀을 하나씩 실행합니다. 셀 왼쪽의 실행 단추를 누르거나 `Shift + Enter` 를 누릅니다.
2. 코드 안의 `____` 는 직접 채우는 빈칸입니다. 빈칸은 다섯 곳이며, 채운 뒤 실행하면 확인 메시지가 나타납니다.
3. **예상 기록** 칸은 결과를 보기 전에 먼저 적습니다. 글 셀을 두 번 누르면 편집할 수 있습니다.
4. 기록을 저장하려면 `파일 > Drive에 사본 저장` 을 먼저 선택합니다.

## 사용하는 데이터
scikit-learn에 포함된 손글씨 숫자 자료(UCI Optical Recognition of Handwritten Digits, 8x8 픽셀, 1,797장)를 사용합니다. 사람의 얼굴, 이름과 위치 정보가 없으며 파일을 올릴 필요가 없습니다. **어두운 사진은 실제 촬영본이 아니라 코드로 밝기를 낮추고 잡음을 더해 만든 조건** 입니다. 따라서 결과는 조명 조건에 따른 기술적 차이를 보여 주며, 특정 사람 집단의 차별을 증명하지 않습니다.

## 0단계, 준비 코드 실행
아래 셀은 실습에 필요한 도구와 확인 함수를 불러옵니다. 내용을 고치지 않고 실행만 합니다.

In [ ]:
# @title 준비 코드 (실행만 하세요) { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 2026
pd.set_option('display.precision', 3)

def _blank_handler(self, etype, value, tb, tb_offset=None):
    history = self.user_ns.get('In') or ['']
    if '____' in history[-1]:
        print('[빈칸 확인] 아직 채우지 않은 빈칸 ____ 이 있습니다. 빈칸을 채운 뒤 다시 실행하세요.')
        return None
    return self.showtraceback((etype, value, tb), tb_offset=tb_offset)

get_ipython().set_custom_exc((Exception,), _blank_handler)

def make_dark(images, seed):
    """밝기를 25%로 낮추고 잡음을 더해 어두운 교실에서 찍은 사진 조건을 만듭니다."""
    rng = np.random.default_rng(seed)
    return np.clip(images * 0.25 + rng.normal(0, 0.10, images.shape), 0, 1)

def build_training_set(dark_ratio, seed=SEED):
    """같은 600장에서 dark_ratio 비율만큼만 어두운 조건으로 바꿉니다."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(X_train_base))
    n_dark = int(round(len(X_train_base) * dark_ratio))
    X = X_train_base.copy()
    dark_idx = order[:n_dark]
    X[dark_idx] = make_dark(X_train_base[dark_idx], seed + 1)
    return X, y_train_base.copy(), n_dark

def train_model(X, y):
    return LogisticRegression(max_iter=3000).fit(X, y)

def evaluate(model, name, X_bright, X_dark, y_true):
    rows = []
    for condition, X in (('밝음', X_bright), ('어두움', X_dark)):
        proba = model.predict_proba(X)
        pred = model.classes_[proba.argmax(axis=1)]
        rows.append(pd.DataFrame({'모델': name, '조건': condition, '정답': y_true,
                                  '예측': pred, '점수': proba.max(axis=1)}))
    df = pd.concat(rows, ignore_index=True)
    df['맞음'] = df['정답'] == df['예측']
    return df

def check(no, value=None):
    ok = '[확인 완료] '
    if no == 1:
        v = value
        if not isinstance(v, (int, float)) or not 0 <= v <= 1:
            print('[다시 확인] 0과 1 사이의 비율을 숫자로 적습니다. 예를 들어 절반은 0.5입니다.')
        elif v <= dark_ratio_A:
            print(f'[다시 확인] 모델 A의 비율({dark_ratio_A})보다 커야 대표성을 높인 비교가 됩니다.')
        elif v > 0.8:
            print(ok + f'모델 B의 어두운 사진 비율은 {v:.0%}입니다. 밝은 사진이 너무 적어지면 밝은 조건의 성능이 달라질 수 있습니다.')
        else:
            print(ok + f'모델 B의 어두운 사진 비율은 {v:.0%}입니다. 두 모델은 같은 600장을 쓰고 어두운 조건의 비율만 다릅니다.')
    elif no == 2:
        expected = accuracy_score(y_final, model_A.predict(X_final))
        if isinstance(value, float) and abs(value - expected) < 1e-12:
            print(ok + f'밝은 사진만으로 계산한 정확도는 {value:.1%}입니다. 판매 자료의 수치와 비슷합니다.')
        else:
            print('[다시 확인] 두 번째 자리에는 모델이 예측한 값이 담긴 변수 이름을 적습니다.')
    elif no == 3:
        if isinstance(value, pd.Series) and list(value.index.names) == ['모델', '조건']:
            print(ok + '모델과 조건으로 나누어 정확도를 계산했습니다. 전체 정확도에 가려진 차이를 비교해 봅니다.')
        else:
            print("[다시 확인] 밝음과 어두움이 적힌 열의 이름을 따옴표 안에 적습니다.")
    elif no == 4:
        dark = results[(results['모델'] == 'A') & (results['조건'] == '어두움')]
        expected = confusion_matrix(dark['정답'], dark['예측'], labels=range(10))
        if isinstance(value, np.ndarray) and value.shape == (10, 10) and (value == expected).all():
            print(ok + '행은 실제 숫자, 열은 모델이 예측한 숫자입니다. 대각선 밖의 칸이 오류입니다.')
        else:
            print("[다시 확인] 두 번째 자리에는 모델이 예측한 숫자가 담긴 열의 이름을 적습니다.")
    elif no == 5:
        if isinstance(value, (int, float)) and 0.5 <= value < 1:
            print(ok + f'점수 {value}보다 낮은 사진은 자동 처리하지 않고 사람이 확인합니다.')
        else:
            print('[다시 확인] 0.5 이상 1 미만의 값을 적습니다. 예를 들어 0.8입니다.')

digits = load_digits()
X_all = digits.images.reshape(len(digits.images), -1) / 16
y_all = digits.target
print(f'준비 완료, 전체 {len(X_all)}장, 숫자 0부터 9까지 10개 클래스')

## 1단계, 밝은 사진과 어두운 사진 살펴보기
같은 숫자를 밝은 조건과 어두운 조건으로 나란히 표시합니다. 사람 눈으로는 어떤 차이가 보이나요?

In [ ]:
sample = X_all[:8]
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(sample[i].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(make_dark(sample[i:i+1], 1).reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(str(y_all[i]))
    for r in range(2):
        axes[r, i].axis('off')
axes[0, 0].text(-6, 4, 'bright', fontsize=11)
axes[1, 0].text(-6, 4, 'dark', fontsize=11)
plt.show()

## 2단계, 학습, 개발, 최종 평가 자료 나누기
학습에 쓴 사진으로 성능을 평가하면 실제보다 좋게 보입니다. 자료를 세 묶음으로 나눕니다.

| 자료 | 용도 |
|---|---|
| 학습 자료 600장 | 모델 A와 B가 똑같이 사용합니다. 어두운 사진의 비율만 다릅니다. |
| 개발 자료 | 판단 보류 기준을 고를 때만 사용합니다. |
| 최종 평가 자료 | 기준을 모두 정한 뒤 한 번만 비교합니다. 같은 사진을 밝음과 어두움으로 한 번씩 평가합니다. |

In [ ]:
X_pool, X_rest, y_pool, y_rest = train_test_split(X_all, y_all, test_size=0.4, stratify=y_all, random_state=SEED)
X_dev, X_final, y_dev, y_final = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=SEED)

rng = np.random.default_rng(SEED)
pick = rng.choice(len(X_pool), 600, replace=False)
X_train_base, y_train_base = X_pool[pick], y_pool[pick]

X_dev_dark = make_dark(X_dev, SEED + 10)
X_final_dark = make_dark(X_final, SEED + 20)

print(f'학습 자료 {len(X_train_base)}장, 개발 자료 {len(X_dev)}장, 최종 평가 자료 {len(X_final)}장')
print(f'최종 평가는 밝음 {len(X_final)}장과 어두움 {len(X_final_dark)}장, 합계 {len(X_final) * 2}회')

## 3단계, 두 모델의 학습 자료 정하기

모델 A는 밝은 사진 위주로 모은 자료입니다. 어두운 사진은 5%뿐입니다.
모델 B는 어두운 교실의 사진을 충분히 포함하도록 수집 방법을 바꾼 자료입니다.

### 예상 기록 (결과를 보기 전에 적기)
- 어두운 사진에서 더 정확한 모델은 A와 B 중 무엇일까요?
- 밝은 사진의 정확도는 어떻게 달라질까요?
- 그렇게 예상한 이유:

In [ ]:
dark_ratio_A = 0.05

# 빈칸 1, 모델 B의 학습 자료에서 어두운 사진의 비율을 0과 1 사이로 적습니다.
dark_ratio_B = ____

check(1, dark_ratio_B)

In [ ]:
X_train_A, y_train_A, dark_A = build_training_set(dark_ratio_A)
X_train_B, y_train_B, dark_B = build_training_set(dark_ratio_B)

summary = pd.DataFrame({
    '모델': ['A', 'B'],
    '밝음': [600 - dark_A, 600 - dark_B],
    '어두움': [dark_A, dark_B],
    '합계': [600, 600],
})
display(summary.style.hide(axis='index'))

model_A = train_model(X_train_A, y_train_A)
model_B = train_model(X_train_B, y_train_B)
print('두 모델의 학습이 끝났습니다. 모델의 종류와 설정은 같고 학습 자료의 조명 구성만 다릅니다.')

## 4단계, 판매 자료의 정확도 확인하기
판매 자료의 정확도는 밝은 사진으로만 계산한 값일 수 있습니다. 모델 A를 밝은 최종 평가 자료로만 평가해 봅니다.

In [ ]:
pred_A_bright = model_A.predict(X_final)

# 빈칸 2, 정확도 = 정답과 예측이 일치한 비율입니다. 예측 결과가 담긴 변수 이름을 적습니다.
acc_A_bright = accuracy_score(y_final, ____)

print(f'모델 A, 밝은 사진만의 정확도 {acc_A_bright:.1%}')
check(2, acc_A_bright)

## 5단계, 조건별 정확도 비교하기
이제 두 모델을 밝음과 어두움 조건에서 모두 평가합니다. 먼저 전체 정확도를 보고, 이어서 조건별로 나누어 봅니다.

In [ ]:
results = pd.concat([
    evaluate(model_A, 'A', X_final, X_final_dark, y_final),
    evaluate(model_B, 'B', X_final, X_final_dark, y_final),
], ignore_index=True)

print('전체 정확도 (밝음과 어두움을 합친 값)')
display(results.groupby('모델')['맞음'].mean().to_frame('전체 정확도'))

In [ ]:
# 빈칸 3, 모델별로 한 번 더 나누어 볼 열의 이름을 적습니다. 밝음과 어두움이 적힌 열입니다.
by_condition = results.groupby(['모델', ____])['맞음'].mean()

display(by_condition.unstack().style.format('{:.1%}'))
check(3, by_condition)

by_condition.unstack().plot(kind='bar', rot=0, ylim=(0, 1), figsize=(6, 3.5))
plt.xlabel('model'); plt.ylabel('accuracy'); plt.legend(['bright', 'dark']); plt.title('Accuracy by lighting condition')
plt.show()

### 해석 기록
- 전체 정확도만 보았을 때와 조건별로 나누었을 때 판단이 어떻게 달라지나요?
- 모델 B에서 어두운 조건의 정확도는 얼마나 달라졌나요? 밝은 조건의 정확도는 어떻게 되었나요?
- 모델 B에도 남은 조건 간 차이는 무엇으로 설명할 수 있을까요?

## 6단계, 어떤 숫자를 틀리는지 살펴보기
정확도는 한 개의 숫자로 요약합니다. 혼동행렬은 **어떤 숫자를 어떤 숫자로 잘못 읽었는지** 보여 줍니다. 모델 A의 어두운 조건을 살펴봅니다.

In [ ]:
dark_result_A = results[(results['모델'] == 'A') & (results['조건'] == '어두움')]

# 빈칸 4, 혼동행렬은 (실제 정답, 모델의 예측)을 비교합니다. 예측이 담긴 열의 이름을 적습니다.
cm = confusion_matrix(dark_result_A['정답'], dark_result_A[____], labels=range(10))

check(4, cm)
plt.figure(figsize=(5, 4.5))
plt.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Model A, dark condition')
plt.show()

errors = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j and cm[i, j] > 0]
for n, i, j in sorted(errors, reverse=True)[:3]:
    print(f'실제 {i}을(를) {j}(으)로 읽은 경우 {n}장')

## 7단계, 판단 보류 기준 정하기
모델은 예측과 함께 점수(가장 높은 확률)를 냅니다. 점수가 낮은 사진은 자동으로 처리하지 않고 사람이 확인하도록 보류할 수 있습니다. 기준은 **개발 자료** 로만 고르고, 최종 평가 자료는 기준을 정한 뒤에 확인합니다.

In [ ]:
dev_results = evaluate(model_B, 'B', X_dev, X_dev_dark, y_dev)
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    auto = dev_results[dev_results['점수'] >= t]
    rows.append({'기준': t, '자동 처리 비율': len(auto) / len(dev_results),
                 '자동 처리 정확도': auto['맞음'].mean() if len(auto) else float('nan'),
                 '사람 검토 장수': len(dev_results) - len(auto)})
display(pd.DataFrame(rows).style.hide(axis='index').format({'기준': '{:.2f}', '자동 처리 비율': '{:.1%}', '자동 처리 정확도': '{:.1%}'}, na_rep='자동 처리 없음'))

In [ ]:
# 빈칸 5, 개발 자료의 표를 보고 자동 처리할 최소 점수를 0.5 이상 1 미만으로 적습니다.
threshold = ____

check(5, threshold)
final_B = results[results['모델'] == 'B']
table = []
for condition, part in final_B.groupby('조건'):
    auto = part[part['점수'] >= threshold]
    table.append({'조건': condition, '평가 장수': len(part), '자동 처리 비율': len(auto) / len(part),
                  '자동 처리 정확도': auto['맞음'].mean() if len(auto) else float('nan'),
                  '사람 검토 장수': len(part) - len(auto), '자동 처리 오류': int((~auto['맞음']).sum())})
display(pd.DataFrame(table).style.hide(axis='index').format({'자동 처리 비율': '{:.1%}', '자동 처리 정확도': '{:.1%}'}, na_rep='자동 처리 없음'))

### 해석 기록
- 보류 기준을 높이면 자동 처리 정확도와 사람의 검토 부담은 각각 어떻게 달라지나요?
- 검토 부담은 어느 조건에 몰리나요? 그 부담을 누가, 어떤 자료로 판단해야 할까요?
- 자동 처리 정확도와 전체 정확도는 분모가 다릅니다. 두 값을 같은 성능 지표로 비교해도 될까요?

## 8단계, 한 번의 결과를 얼마나 믿을 수 있을까
학습 자료를 고르는 무작위 순서만 바꾸어 같은 실험을 다섯 번 반복합니다. 조건 간 차이가 반복해서 나타나는지 확인합니다.

In [ ]:
def run_once(seed):
    global X_train_base, y_train_base
    r = np.random.default_rng(seed)
    idx = r.choice(len(X_pool), 600, replace=False)
    X_train_base, y_train_base = X_pool[idx], y_pool[idx]
    out = {'반복': seed - SEED + 1}
    for name, ratio in (('A', dark_ratio_A), ('B', dark_ratio_B)):
        Xtr, ytr, _ = build_training_set(ratio, seed)
        m = train_model(Xtr, ytr)
        out[f'{name} 밝음'] = accuracy_score(y_final, m.predict(X_final))
        out[f'{name} 어두움'] = accuracy_score(y_final, m.predict(X_final_dark))
    return out

base_backup = (X_train_base, y_train_base)
repeat = pd.DataFrame([run_once(SEED + k) for k in range(5)]).set_index('반복')
X_train_base, y_train_base = base_backup
display(repeat.style.format('{:.1%}'))
print('평균')
display(repeat.mean().to_frame('평균').T.style.format('{:.1%}'))
print('표준편차')
display(repeat.std().to_frame('표준편차').T.style.format('{:.1%}'))

## 9단계, 윤리 검토 기록
실험 결과를 근거로 도구의 사용 여부를 판단합니다. 확인하지 못한 사항은 추측으로 채우지 않고 **미확인** 으로 남깁니다.

| 항목 | 기록 |
|---|---|
| 데이터의 목적과 범위 | |
| 자료 구성 (조명 조건, 장수, 분리 방식) | |
| 조건별 성능과 표본 수 | |
| 보류 기준과 검토 담당자 | |
| 검증하지 않은 조건 (실제 촬영 사진, 다른 필체, 다른 기기) | |
| 개인정보와 자료 권한 검토 | |
| 판단 (제한된 사용, 조건부 사용, 보류) 과 근거 | |

## 선택 확장, 어두운 사진 비율을 바꾸어 보기
어두운 사진의 비율을 0%에서 90%까지 바꾸며 두 조건의 정확도가 어떻게 달라지는지 그래프로 확인합니다. 계산에 1분 정도 걸릴 수 있습니다.

In [ ]:
ratios = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
curve = []
for ratio in ratios:
    Xtr, ytr, _ = build_training_set(ratio)
    m = train_model(Xtr, ytr)
    curve.append({'ratio': ratio, 'bright': accuracy_score(y_final, m.predict(X_final)),
                  'dark': accuracy_score(y_final, m.predict(X_final_dark))})
curve = pd.DataFrame(curve).set_index('ratio')
curve.index = [f'{r:.0%}' for r in curve.index]
curve.plot(marker='o', ylim=(0, 1), figsize=(6, 3.5))
plt.xticks(range(len(ratios)), curve.index); plt.xlabel('share of dark images in training data'); plt.ylabel('accuracy'); plt.show()
display(curve.style.format('{:.1%}'))

---
자료 출처: scikit-learn `load_digits` (UCI Optical Recognition of Handwritten Digits, CC BY 4.0). 어두운 조건은 실습을 위해 코드로 만든 변환이며 실제 교실 사진의 측정 결과가 아닙니다.